# Бейзлайн: LFCC + GMM и CQCC + GMM

Отправная точка проекта — воспроизведение официальных бейзлайнов ASVspoof2019
(B01: CQCC-GMM, B02: LFCC-GMM). На эти два пайплайна дальше будем опираться,
сравнивая с более сложными/эффективными подходами (LCNN, OC-Softmax и т.д.)
в отдельных ноутбуках.

**Идея бейзлайна:** для каждого типа признаков обучаются две GMM (Gaussian
Mixture Model) — одна на bonafide-примерах, другая на spoof-примерах.
Скор утверждения (utterance-level score) — логарифм отношения правдоподобий:

$$\text{score}(x) = \frac{1}{T}\sum_{t=1}^{T} \log p(x_t \mid \lambda_{bonafide}) - \frac{1}{T}\sum_{t=1}^{T} \log p(x_t \mid \lambda_{spoof})$$

где x₁..x_T — кадры (фреймы) признаков одного аудиофайла. Чем выше score,
тем более речь похожа на настоящую.

**Требование:** нужно, чтобы `02_eda.ipynb` уже был запущен — этот пайплайн
переиспользует `data/processed/protocol_summary.csv`, построенный там.


## 1. Настройка окружения

In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import scipy.fftpack
from sklearn.mixture import GaussianMixture
from tqdm.auto import tqdm
import joblib

sys.path.append(".")  # чтобы находился пакет src/, если ноутбук запущен из корня проекта
from src.common import load_config, load_protocol_with_paths, compute_eer, save_pipeline_results

warnings.filterwarnings("ignore")
%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

config = load_config()
SEED = config["project"]["seed"]
rng = np.random.default_rng(SEED)

SAMPLE_RATE = config["audio"]["sample_rate"]
GMM_CFG = config["baseline_gmm"]

FEATURES_DIR = Path(config["paths"]["features_dir"])
MODELS_DIR = Path(config["paths"]["models_dir"])
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Sample rate: {SAMPLE_RATE}")
print(f"GMM: n_components={GMM_CFG['n_components']}, covariance_type={GMM_CFG['covariance_type']}")


In [ ]:
protocol_df = load_protocol_with_paths(config)
print(f"Всего записей в протоколе: {len(protocol_df)}")
protocol_df.groupby(["split", "label"]).size().unstack(fill_value=0)


## 2. Извлечение признаков

### LFCC (Linear Frequency Cepstral Coefficients)

Пайплайн: STFT → степенной спектр → **линейный** (равномерный по Гц, в отличие
от MFCC) банк треугольных фильтров → логарифм энергии → DCT → статические
коэффициенты + Δ (delta) + ΔΔ (delta-delta).


In [ ]:
def linear_filterbank(sr: int, n_fft: int, n_filters: int) -> np.ndarray:
    """Треугольный банк фильтров, равномерно расположенных по линейной шкале Гц
    (в отличие от mel-шкалы, используемой в MFCC) — то, что делает признак LFCC."""
    freqs = np.linspace(0, sr / 2, n_filters + 2)
    bins = np.floor((n_fft + 1) * freqs / sr).astype(int)
    fbank = np.zeros((n_filters, n_fft // 2 + 1))

    for m in range(1, n_filters + 1):
        f_left, f_center, f_right = bins[m - 1], bins[m], bins[m + 1]
        f_center = max(f_center, f_left + 1)
        f_right = max(f_right, f_center + 1)
        for k in range(f_left, f_center):
            fbank[m - 1, k] = (k - f_left) / (f_center - f_left)
        for k in range(f_center, min(f_right, fbank.shape[1])):
            fbank[m - 1, k] = (f_right - k) / (f_right - f_center)

    return fbank


def extract_lfcc(y: np.ndarray, sr: int, n_fft: int, hop_length: int, win_length: int,
                  n_filters: int, n_ceps: int) -> np.ndarray:
    stft = librosa.stft(y, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window="hamming")
    power_spec = np.abs(stft) ** 2

    fbank = linear_filterbank(sr, n_fft, n_filters)
    filtered = fbank @ power_spec
    filtered = np.maximum(filtered, 1e-10)
    log_energy = np.log(filtered)

    static = scipy.fftpack.dct(log_energy, type=2, axis=0, norm="ortho")[:n_ceps]
    delta = librosa.feature.delta(static)
    delta2 = librosa.feature.delta(static, order=2)

    return np.vstack([static, delta, delta2])  # (3*n_ceps, n_frames)


### CQCC (Constant-Q Cepstral Coefficients)

Пайплайн: Constant-Q Transform (CQT, геометрическая/логарифмическая шкала
частот вместо линейной) → логарифм степенного спектра → DCT → статические
коэффициенты + Δ + ΔΔ.

**Важная оговорка:** это практическое воспроизведение общей идеи алгоритма
Todisco et al. (2016), а не побитовая копия оригинального MATLAB-тулкита —
там используется дополнительная схема ресемплинга между несколькими
разрешениями по октавам. Здесь применена облегчённая конфигурация
(`bins_per_octave=12` вместо 96 в оригинальной статье) — сознательный
компромисс в пользу скорости и памяти, обсуждавшийся ранее.


In [ ]:
def extract_cqcc(y: np.ndarray, sr: int, n_bins: int, bins_per_octave: int,
                  hop_length: int, n_ceps: int) -> np.ndarray:
    # Защита от выхода за границу Найквиста при неудачном сочетании n_bins/bins_per_octave
    # в config.yaml (librosa иначе выбросит ValueError на КАЖДОМ файле).
    max_octaves = np.log2((sr / 2) / librosa.note_to_hz("C1")) - 0.15  # запас ~0.15 октавы
    safe_n_bins = min(n_bins, int(max_octaves * bins_per_octave))
    if safe_n_bins < n_bins:
        n_bins = safe_n_bins

    cqt = librosa.cqt(y, sr=sr, hop_length=hop_length, n_bins=n_bins, bins_per_octave=bins_per_octave)
    power_spec = np.abs(cqt) ** 2
    power_spec = np.maximum(power_spec, 1e-10)
    log_power = np.log(power_spec)

    static = scipy.fftpack.dct(log_power, type=2, axis=0, norm="ortho")[:n_ceps]
    delta = librosa.feature.delta(static)
    delta2 = librosa.feature.delta(static, order=2)

    return np.vstack([static, delta, delta2])  # (3*n_ceps, n_frames)


## 3. Извлечение с кэшированием на диск

Каждый файл обрабатывается один раз — результат сохраняется в
`data/features/<тип_признака>/<имя_файла>.npy`, при повторном запуске
ноутбука признаки просто подгружаются с диска.


In [ ]:
def extract_features_cached(path: Path, feature_type: str, sr: int, params: dict) -> np.ndarray:
    cache_path = FEATURES_DIR / feature_type / f"{path.stem}.npy"
    if cache_path.exists():
        return np.load(cache_path)

    y, _ = librosa.load(path, sr=sr)

    if feature_type == "lfcc":
        feats = extract_lfcc(y, sr, **params)
    elif feature_type == "cqcc":
        feats = extract_cqcc(y, sr, **params)
    else:
        raise ValueError(f"Неизвестный тип признака: {feature_type}")

    cache_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(cache_path, feats)
    return feats


def extract_split_features(df: pd.DataFrame, split: str, label: str, feature_type: str,
                            params: dict, max_files=None, desc_prefix=""):
    subset = df[(df["split"] == split) & (df["label"] == label) & df["full_path"].notna()]
    if max_files is not None and len(subset) > max_files:
        subset = subset.sample(n=max_files, random_state=SEED)

    feats_list = []
    desc = f"{desc_prefix}{feature_type} | {split}/{label}"
    for _, row in tqdm(subset.iterrows(), total=len(subset), desc=desc):
        try:
            feats_list.append(extract_features_cached(row["full_path"], feature_type, SAMPLE_RATE, params))
        except Exception as e:
            print(f"  Пропуск {row['filename']}: {e}")

    return feats_list, subset


## 4. Обучение GMM (для LFCC и CQCC отдельно)

Для каждого типа признаков — своя пара GMM (bonafide / spoof), обученная на
train-сплите. Кадры всех файлов одного класса объединяются в одну матрицу,
затем (при необходимости) подвыбираются до `max_train_frames_per_class`,
чтобы обучение не упиралось в память.


In [ ]:
def stack_and_subsample_frames(feats_list, max_frames=None, seed=SEED):
    all_frames = np.concatenate([f.T for f in feats_list], axis=0)  # (total_frames, dim)
    if max_frames is not None and len(all_frames) > max_frames:
        idx = np.random.default_rng(seed).choice(len(all_frames), size=max_frames, replace=False)
        all_frames = all_frames[idx]
    return all_frames


def safe_n_components(n_samples: int, desired: int) -> int:
    """На маленьких (например, демо) данных 512 компонент GMM обучить нельзя —
    автоматически уменьшаем, чтобы не падать с ошибкой sklearn."""
    max_reasonable = max(1, n_samples // 10)
    return min(desired, max_reasonable)


def train_gmm(frames: np.ndarray, n_components: int, covariance_type: str, reg_covar: float, max_iter: int, seed=SEED):
    n_components = safe_n_components(len(frames), n_components)
    gmm = GaussianMixture(
        n_components=n_components,
        covariance_type=covariance_type,
        reg_covar=reg_covar,
        max_iter=max_iter,
        random_state=seed,
    )
    gmm.fit(frames)
    return gmm


In [ ]:
FEATURE_PARAMS = {
    "lfcc": GMM_CFG["lfcc"],
    "cqcc": GMM_CFG["cqcc"],
}

trained_gmms = {}       # {feature_type: {"bonafide": gmm, "spoof": gmm}}
train_feature_cache = {}  # чтобы не извлекать признаки для train повторно на след. шагах

for feature_type, params in FEATURE_PARAMS.items():
    print(f"\n=== Обучение GMM на признаках: {feature_type.upper()} ===")
    trained_gmms[feature_type] = {}

    for label in ("bonafide", "spoof"):
        feats_list, _ = extract_split_features(
            protocol_df, split="train", label=label, feature_type=feature_type,
            params=params, max_files=GMM_CFG["max_train_files_per_class"],
        )
        train_feature_cache[(feature_type, label)] = feats_list

        frames = stack_and_subsample_frames(feats_list, max_frames=GMM_CFG["max_train_frames_per_class"])
        print(f"  {label}: файлов={len(feats_list)}, кадров для обучения={len(frames)}, размерность={frames.shape[1]}")

        gmm = train_gmm(
            frames,
            n_components=GMM_CFG["n_components"],
            covariance_type=GMM_CFG["covariance_type"],
            reg_covar=GMM_CFG["reg_covar"],
            max_iter=GMM_CFG["max_iter"],
        )
        trained_gmms[feature_type][label] = gmm

        model_path = MODELS_DIR / f"gmm_{feature_type}_{label}.joblib"
        joblib.dump(gmm, model_path)
        print(f"  Модель сохранена: {model_path} (n_components={gmm.n_components})")


## 5. Скоринг dev/eval и расчёт EER


In [ ]:
def score_utterances(feats_list, gmm_bona, gmm_spoof):
    scores = []
    for feats in feats_list:
        x = feats.T
        ll_bona = gmm_bona.score_samples(x).mean()
        ll_spoof = gmm_spoof.score_samples(x).mean()
        scores.append(ll_bona - ll_spoof)
    return np.array(scores)


eval_results = {}  # {feature_type: {"dev": {...}, "eval": {...}}}

for feature_type, params in FEATURE_PARAMS.items():
    print(f"\n=== Скоринг: {feature_type.upper()} ===")
    eval_results[feature_type] = {}
    gmm_bona = trained_gmms[feature_type]["bonafide"]
    gmm_spoof = trained_gmms[feature_type]["spoof"]

    for split in ("dev", "eval"):
        bona_feats, _ = extract_split_features(
            protocol_df, split=split, label="bonafide", feature_type=feature_type,
            params=params, max_files=GMM_CFG["max_eval_files_per_class"],
        )
        spoof_feats, _ = extract_split_features(
            protocol_df, split=split, label="spoof", feature_type=feature_type,
            params=params, max_files=GMM_CFG["max_eval_files_per_class"],
        )

        bona_scores = score_utterances(bona_feats, gmm_bona, gmm_spoof)
        spoof_scores = score_utterances(spoof_feats, gmm_bona, gmm_spoof)

        eer, threshold = compute_eer(bona_scores, spoof_scores)
        eval_results[feature_type][split] = {
            "eer": eer,
            "threshold": threshold,
            "bona_scores": bona_scores,
            "spoof_scores": spoof_scores,
            "n_bonafide": len(bona_scores),
            "n_spoof": len(spoof_scores),
        }
        print(f"  {split}: EER = {eer * 100:.2f}%  (bonafide={len(bona_scores)}, spoof={len(spoof_scores)})")


## 6. Сводная таблица результатов

In [ ]:
summary_rows = []
for feature_type, splits in eval_results.items():
    for split, res in splits.items():
        summary_rows.append({
            "feature": feature_type.upper(),
            "split": split,
            "EER_%": round(res["eer"] * 100, 2),
            "n_bonafide": res["n_bonafide"],
            "n_spoof": res["n_spoof"],
        })

summary_df = pd.DataFrame(summary_rows).sort_values(["feature", "split"])
summary_df


## 7. Распределение скоров (dev): bonafide vs spoof

In [ ]:
fig, axes = plt.subplots(1, len(FEATURE_PARAMS), figsize=(12, 4))
if len(FEATURE_PARAMS) == 1:
    axes = [axes]

for ax, (feature_type, res) in zip(axes, eval_results.items()):
    dev = res["dev"]
    ax.hist(dev["bona_scores"], bins=20, alpha=0.6, label="bonafide", color="#4C72B0")
    ax.hist(dev["spoof_scores"], bins=20, alpha=0.6, label="spoof", color="#C44E52")
    ax.axvline(dev["threshold"], color="black", linestyle="--", linewidth=1, label="порог EER")
    ax.set_title(f"{feature_type.upper()} — dev (EER={dev['eer']*100:.2f}%)")
    ax.set_xlabel("score = ll(bonafide) - ll(spoof)")
    ax.set_ylabel("Количество файлов")
    ax.legend()

plt.tight_layout()
plt.show()


## 8. Сохранение результатов для итогового сравнения пайплайнов

Результат пишется в `results/lfcc_gmm.json` и `results/cqcc_gmm.json` —
единый формат, который позднее соберёт в одну таблицу финальный ноутбук
сравнения всех пайплайнов (LFCC-GMM, CQCC-GMM, LCNN, ... ).


In [ ]:
for feature_type, splits in eval_results.items():
    metrics = {
        "feature_type": feature_type,
        "backend": "GMM",
        "n_components": trained_gmms[feature_type]["bonafide"].n_components,
        "covariance_type": GMM_CFG["covariance_type"],
        "eer_dev_%": round(splits["dev"]["eer"] * 100, 2),
        "eer_eval_%": round(splits["eval"]["eer"] * 100, 2),
    }
    save_pipeline_results(name=f"{feature_type}_gmm", metrics=metrics, config=config)


## Дальнейшие шаги

1. При необходимости повысить `bins_per_octave` для CQCC и/или `n_components`
   GMM в `config.yaml` для более честного воспроизведения оригинального бейзлайна
   (потребует больше времени/памяти — сейчас конфигурация облегчена намеренно).
2. Следующий пайплайн (например, LCNN) оформить отдельным ноутбуком
   `04_lcnn.ipynb`, переиспользуя `src/common.py` для конфига, протокола, EER и
   сохранения результатов — так же, как сделано здесь.
3. Когда будут готовы 2–3 пайплайна, финальный ноутбук сравнения соберёт
   все `results/*.json` через `load_all_pipeline_results(config)` в одну таблицу.
